# Två agenter i lagret

# Warehouse grid

Tecken i kartan:
* `.` golv
* `#` hylla
* `S` start
* `G` mål

Koordinater är (rad, kolumn), där (0, 0) är övre vänstra hörnet.

In [1]:
from collections import deque

import numpy as np

In [2]:
def make_layout(
    n_shelf_rows=2,
    shelves_per_row=3,
    shelf_width=2,
    shelf_depth=2,
):
    # en gång mer än antalet hyllor, åt båda hållen
    n_rows = n_shelf_rows * shelf_depth + (n_shelf_rows + 1)
    n_cols = shelves_per_row * shelf_width + (shelves_per_row + 1)

    # börja med golv överallt
    grid = np.full((n_rows, n_cols), ".")

    # lägg in hyllorna
    for i in range(n_shelf_rows):
        for j in range(shelves_per_row):
            top = 1 + i * (shelf_depth + 1)
            left = 1 + j * (shelf_width + 1)
            grid[top:top + shelf_depth, left:left + shelf_width] = "#"

    # start uppe till vänster, mål nere till höger
    grid[0, 0] = "S"
    grid[-1, -1] = "G"

    return "\n".join("".join(row) for row in grid)

In [3]:
class WarehouseGrid:
    def __init__(self, layout=None):
        # standardkartan om ingen annan skickas in
        if layout is None:
            layout = make_layout()

        rows = layout.strip().splitlines()
        self.n_rows = len(rows)
        self.n_cols = len(rows[0])

        # vilket tecken som står i varje cell
        symbols = {(r, c): symbol
                   for r, row in enumerate(rows)
                   for c, symbol in enumerate(row)}

        # sortera cellerna efter tecken
        self.shelf_cells = {cell for cell, s in symbols.items() if s == "#"}
        self.walkable_cells = {cell for cell, s in symbols.items() if s != "#"}
        self.start = next((cell for cell, s in symbols.items() if s == "S"), None)
        self.goal = next((cell for cell, s in symbols.items() if s == "G"), None)

        # plockplatser: golvcellerna bredvid varje hyllcell
        self.pick_spots = {
            shelf: self.walkable_neighbors(shelf) for shelf in self.shelf_cells
        }

        # vilken vara som ligger på vilken hylla
        self.item_to_shelf = {}

        self.check_all_cells_reachable()

    # golvcellerna ovanför, under, till vänster och till höger
    def walkable_neighbors(self, cell):
        r, c = cell
        around = [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)]
        return [n for n in around if n in self.walkable_cells]

    # felmeddelande om världen är uppdelad eller en hylla inte går att nå
    def check_all_cells_reachable(self):
        start = next(iter(self.walkable_cells))
        visited = {start}
        queue = deque([start])
        while queue:
            for n in self.walkable_neighbors(queue.popleft()):
                if n not in visited:
                    visited.add(n)
                    queue.append(n)

        # celler som aldrig nåddes betyder att världen är uppdelad
        if visited != self.walkable_cells:
            raise ValueError(f"Här kan du inte gå för fan! {sorted(self.walkable_cells - visited)}")

        # varje hylla måste ha minst en plockplats
        for shelf, spots in self.pick_spots.items():
            if not spots:
                raise ValueError(f"Hyllan {shelf} har ingen plockplats.")

    # lägg en vara på en hylla
    def add_item(self, item, shelf):
        if shelf not in self.shelf_cells:
            raise ValueError(f"{shelf} är ingen hylla!")
        self.item_to_shelf[item] = shelf

    # lägg varje vara på en slumpad hylla, max en vara per hylla
    def place_items_randomly(self, items, seed=None):
        rng = np.random.default_rng(seed)
        shelves = sorted(self.shelf_cells)
        chosen = rng.choice(len(shelves), size=len(items), replace=False)
        for item, i in zip(items, chosen):
            self.add_item(item, shelves[i])

    # var man ska stå för att plocka en vara
    def pick_spots_for_item(self, item):
        return self.pick_spots[self.item_to_shelf[item]]

    # rita världen, lager för lager
    def render(self, show_pick_spots=False):
        grid = np.full((self.n_rows, self.n_cols), ".")

        if show_pick_spots:
            for spots in self.pick_spots.values():
                for cell in spots:
                    grid[cell] = "p"

        for cell in self.shelf_cells:
            grid[cell] = "#"

        if self.start is not None:
            grid[self.start] = "S"
        if self.goal is not None:
            grid[self.goal] = "G"

        for row in grid:
            print(" ".join(row))
        print()

## Steg 1 och 2: två agenter som kan krocka

* Varje agent har en egen start och ett eget mål.
* `WAIT` = stå still, så att en agent kan släppa förbi den andra.
* Staten är båda agenternas positioner: `((rad, kol), (rad, kol))`.

Regler i `step()`:
1. Hylla eller utanför kartan -> agenten står kvar.
2. Båda vill till samma ruta -> båda står kvar.
3. De försöker byta plats -> båda står kvar.

Varje steg ger `-1`. Den som blir stoppad av den andra agenten får `-5` extra. En agent i mål står still och får `0`.

In [ ]:
STEP_REWARD = -1    # varje steg kostar, så agenterna vill bli klara snabbt
CRASH_REWARD = -5   # extra minus om man försöker gå in i den andra agenten

# hur varje handling flyttar en agent (rad, kolumn)
ACTIONS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
    "WAIT": (0, 0),
}


class TwoAgentWarehouse:
    def __init__(self, grid, starts, goals):
        self.grid = grid        # Jockes WarehouseGrid, hyllor och golv
        self.starts = starts    # en start per agent, t.ex. [(0, 0), (0, 9)]
        self.goals = goals      # ett mål per agent
        self.reset()

    # ställ båda agenterna på sin start igen
    def reset(self):
        self.positions = list(self.starts)
        return tuple(self.positions)

    # rita världen med agenterna A och B, och deras mål a och b
    def render(self):
        grid = np.full((self.grid.n_rows, self.grid.n_cols), ".")

        for cell in self.grid.shelf_cells:
            grid[cell] = "#"

        for goal, name in zip(self.goals, "ab"):
            grid[goal] = name
        for pos, name in zip(self.positions, "AB"):
            grid[pos] = name

        for row in grid:
            print(" ".join(row))
        print()

    # flytta en agent ett steg, stå kvar om det är hylla eller utanför kartan
    def move(self, pos, action):
        dr, dc = ACTIONS[action]
        new = (pos[0] + dr, pos[1] + dc)
        if new in self.grid.walkable_cells:
            return new
        return pos

    # båda agenterna tar en handling samtidigt, t.ex. actions = ("RIGHT", "WAIT")
    def step(self, actions):
        old = list(self.positions)

        # vart vill agenterna gå? en agent som redan är i mål står still
        wanted = []
        for i in range(2):
            if old[i] == self.goals[i]:
                wanted.append(old[i])
            else:
                wanted.append(self.move(old[i], actions[i]))

        # samma ruta eller byta plats med varandra -> båda står kvar
        same_cell = wanted[0] == wanted[1]
        swap = wanted[0] == old[1] and wanted[1] == old[0]
        if same_cell or swap:
            new = old
        else:
            new = wanted

        # rewards, en per agent
        rewards = []
        for i in range(2):
            if old[i] == self.goals[i]:
                rewards.append(0)  # redan klar, inget mer minus
                continue
            reward = STEP_REWARD
            # ville gå men blev stoppad av den andra agenten
            if wanted[i] != old[i] and new[i] == old[i]:
                reward += CRASH_REWARD
            rewards.append(reward)

        self.positions = list(new)
        done = all(p == g for p, g in zip(self.positions, self.goals))
        return tuple(self.positions), rewards, done


In [ ]:
env = TwoAgentWarehouse(
    WarehouseGrid(),
    starts=[(0, 0), (0, 9)],
    goals=[(6, 9), (6, 0)],
)
print(env.reset())
env.render()


((0, 0), (0, 9))
A . . . . . . . . B
. # # . # # . # # .
. # # . # # . # # .
. . . . . . . . . .
. # # . # # . # # .
. # # . # # . # # .
b . . . . . . . . a



### Test av krockreglerna

In [ ]:
# en smal korridor, så vi kan tvinga fram krockar
env = TwoAgentWarehouse(
    WarehouseGrid("....."),
    starts=[(0, 0), (0, 4)],
    goals=[(0, 4), (0, 0)],
)
env.render()

tests = [
    (("RIGHT", "LEFT"), "båda går ett steg"),
    (("RIGHT", "LEFT"), "båda vill till mitten -> krock"),
    (("RIGHT", "WAIT"), "B väntar, A går till mitten"),
    (("RIGHT", "LEFT"), "de försöker byta plats -> krock"),
    (("UP", "WAIT"), "A går in i väggen, står kvar men ingen krock"),
]
for actions, text in tests:
    state, rewards, done = env.step(actions)
    print(f"{actions}: {text}")
    print(f"rewards = {rewards}")
    env.render()


A . . . B

('RIGHT', 'LEFT'): båda går ett steg
rewards = [-1, -1]
b A . B a

('RIGHT', 'LEFT'): båda vill till mitten -> krock
rewards = [-6, -6]
b A . B a

('RIGHT', 'WAIT'): B väntar, A går till mitten
rewards = [-1, -1]
b . A B a

('RIGHT', 'LEFT'): de försöker byta plats -> krock
rewards = [-6, -6]
b . A B a

('UP', 'WAIT'): A går in i väggen, står kvar men ingen krock
rewards = [-1, -1]
b . A B a



## Steg 3: agenterna lär sig med Q-learning

* Varje agent har en egen Q-tabell (oberoende Q-learning).
* Staten är båda positionerna, så varje agent vet var den andra står.
* Epsilon-greedy: oftast bästa handlingen, ibland en slumpad för att utforska.
* Uppdatering: `Q[s][a] += alpha * (r + gamma * max Q[s'] - Q[s][a])`

In [ ]:
from collections import defaultdict

ACTION_NAMES = list(ACTIONS)  # ["UP", "DOWN", "LEFT", "RIGHT", "WAIT"]


# välj handling: oftast den bästa, ibland en slumpad (epsilon-greedy)
def choose_action(Q, state, epsilon, rng):
    if rng.random() < epsilon:
        return rng.integers(len(ACTION_NAMES))
    q = Q[state]
    best = np.flatnonzero(q == q.max())  # om flera är lika bra, slumpa bland dem
    return rng.choice(best)


# varje agent har en egen Q-tabell, men ser båda positionerna
def train(env, episodes=10000, alpha=0.5, gamma=0.99, epsilon=0.1, max_steps=200, seed=0):
    rng = np.random.default_rng(seed)
    Qs = [defaultdict(lambda: np.zeros(len(ACTION_NAMES))) for _ in range(2)]
    steps_per_episode = []

    for episode in range(episodes):
        state = env.reset()
        for t in range(max_steps):
            a = [choose_action(Qs[i], state, epsilon, rng) for i in range(2)]
            next_state, rewards, done = env.step((ACTION_NAMES[a[0]], ACTION_NAMES[a[1]]))

            for i in range(2):
                if state[i] == env.goals[i]:
                    continue  # redan i mål, inget att lära
                if next_state[i] == env.goals[i]:
                    target = rewards[i]  # i mål, ingen framtid kvar
                else:
                    target = rewards[i] + gamma * Qs[i][next_state].max()
                Qs[i][state][a[i]] += alpha * (target - Qs[i][state][a[i]])

            state = next_state
            if done:
                break
        steps_per_episode.append(t + 1)

    return Qs, steps_per_episode


In [ ]:
# kör en episod utan slump och visa vad agenterna lärt sig
def run_greedy(env, Qs, max_steps=50, show=True):
    rng = np.random.default_rng(0)
    state = env.reset()
    finished = [None, None]  # vilket steg varje agent kom i mål
    crashes = 0
    if show:
        env.render()

    for t in range(1, max_steps + 1):
        a = [choose_action(Qs[i], state, 0.0, rng) for i in range(2)]
        state, rewards, done = env.step((ACTION_NAMES[a[0]], ACTION_NAMES[a[1]]))
        crashes += sum(r < STEP_REWARD for r in rewards)
        for i in range(2):
            if finished[i] is None and state[i] == env.goals[i]:
                finished[i] = t
        if show:
            print(f"steg {t}: A {ACTION_NAMES[a[0]]}, B {ACTION_NAMES[a[1]]}")
            env.render()
        if done:
            break

    print(f"A i mål efter {finished[0]} steg, B efter {finished[1]} steg, {crashes} krockar")


In [ ]:
env = TwoAgentWarehouse(
    WarehouseGrid(),
    starts=[(0, 0), (0, 9)],
    goals=[(6, 9), (6, 0)],
)
Qs, steps = train(env)

# medel antal steg per 1000 episoder, ska gå ner när de lär sig
for i in range(0, len(steps), 1000):
    print(f"episod {i:4d}-{i + 999}: {np.mean(steps[i:i + 1000]):6.1f} steg")


episod    0-999:  105.4 steg
episod 1000-1999:   48.6 steg
episod 2000-2999:   35.1 steg
episod 3000-3999:   29.3 steg
episod 4000-4999:   27.0 steg
episod 5000-5999:   24.2 steg
episod 6000-6999:   23.0 steg
episod 7000-7999:   20.9 steg
episod 8000-8999:   19.6 steg
episod 9000-9999:   19.7 steg


In [ ]:
run_greedy(env, Qs)

A . . . . . . . . B
. # # . # # . # # .
. # # . # # . # # .
. . . . . . . . . .
. # # . # # . # # .
. # # . # # . # # .
b . . . . . . . . a

steg 1: A DOWN, B LEFT
. . . . . . . . B .
A # # . # # . # # .
. # # . # # . # # .
. . . . . . . . . .
. # # . # # . # # .
. # # . # # . # # .
b . . . . . . . . a

steg 2: A DOWN, B LEFT
. . . . . . . B . .
. # # . # # . # # .
A # # . # # . # # .
. . . . . . . . . .
. # # . # # . # # .
. # # . # # . # # .
b . . . . . . . . a

steg 3: A DOWN, B LEFT
. . . . . . B . . .
. # # . # # . # # .
. # # . # # . # # .
A . . . . . . . . .
. # # . # # . # # .
. # # . # # . # # .
b . . . . . . . . a

steg 4: A DOWN, B LEFT
. . . . . B . . . .
. # # . # # . # # .
. # # . # # . # # .
. . . . . . . . . .
A # # . # # . # # .
. # # . # # . # # .
b . . . . . . . . a

steg 5: A DOWN, B LEFT
. . . . B . . . . .
. # # . # # . # # .
. # # . # # . # # .
. . . . . . . . . .
. # # . # # . # # .
A # # . # # . # # .
b . . . . . . . . a

steg 6: A DOWN, B LEFT
. . . B . . . . 

## Steg 4: trång karta

En korridor med en enda ficka i mitten. Agenterna måste mötas, så en av dem måste väja in i fickan och vänta.

Bästa möjliga: den ena går raka vägen på 8 steg plus lite väntan, den andra väjer. Totalt 10 + 9 steg.

In [ ]:
# en korridor med en enda ficka i mitten där man kan väja
narrow = TwoAgentWarehouse(
    WarehouseGrid("""
####.####
.........
"""),
    starts=[(1, 0), (1, 8)],
    goals=[(1, 8), (1, 0)],
)
narrow_Qs, narrow_steps = train(narrow)
run_greedy(narrow, narrow_Qs)


# # # # . # # # #
A . . . . . . . B

steg 1: A RIGHT, B UP
# # # # . # # # #
b A . . . . . . B

steg 2: A RIGHT, B UP
# # # # . # # # #
b . A . . . . . B

steg 3: A RIGHT, B LEFT
# # # # . # # # #
b . . A . . . B a

steg 4: A RIGHT, B WAIT
# # # # . # # # #
b . . . A . . B a

steg 5: A DOWN, B LEFT
# # # # . # # # #
b . . . A . B . a

steg 6: A UP, B WAIT
# # # # A # # # #
b . . . . . B . a

steg 7: A RIGHT, B LEFT
# # # # A # # # #
b . . . . B . . a

steg 8: A WAIT, B LEFT
# # # # A # # # #
b . . . B . . . a

steg 9: A WAIT, B LEFT
# # # # A # # # #
b . . B . . . . a

steg 10: A WAIT, B LEFT
# # # # A # # # #
b . B . . . . . a

steg 11: A DOWN, B LEFT
# # # # . # # # #
b B . . A . . . a

steg 12: A RIGHT, B LEFT
# # # # . # # # #
B . . . . A . . a

steg 13: A RIGHT, B LEFT
# # # # . # # # #
B . . . . . A . a

steg 14: A RIGHT, B WAIT
# # # # . # # # #
B . . . . . . A a

steg 15: A RIGHT, B RIGHT
# # # # . # # # #
B . . . . . . . A

A i mål efter 15 steg, B efter 12 steg, 0 krockar


Resultatet varierar beroende på seed. Båda lär sig alltid att undvika krockar, men inte alltid den kortaste vägen. Det beror på att varje agent lär sig medan den andra också lär sig och ändrar beteende, så världen ser inte likadan ut från en episod till nästa.

In [ ]:
# samma karta med olika inställningar och seeds, bästa möjliga är 10 och 9 steg
for settings in [{}, {"seed": 1}, {"seed": 2}, {"episodes": 50000, "alpha": 0.2}]:
    test_Qs, _ = train(narrow, **settings)
    print(f"{str(settings):35}", end="")
    run_greedy(narrow, test_Qs, show=False)


{}                                 A i mål efter 15 steg, B efter 12 steg, 0 krockar
{'seed': 1}                        A i mål efter 12 steg, B efter 10 steg, 0 krockar
{'seed': 2}                        A i mål efter 13 steg, B efter 10 steg, 0 krockar
{'episodes': 50000, 'alpha': 0.2}  A i mål efter 9 steg, B efter 10 steg, 0 krockar
